# Iris → PyTorch MLP → ONNX → OpenVINO / RHOAI Model Registry

This notebook replaces the scikit-learn `RandomForestClassifier` ONNX model that uses
`ai.onnx.ml.TreeEnsembleClassifier`, which OpenVINO Model Server does not support.

It trains two small neural-network versions from the same `Iris.csv` dataset:

- **v1**: one hidden layer with 8 neurons
- **v2**: one hidden layer with 12 neurons

Both versions:
- accept the same four raw Iris measurements
- normalize inputs inside the model graph
- return three class probabilities
- export to standard ONNX operators supported by OpenVINO
- keep the same input/output contract for easy Model Registry versioning

Put `Iris.csv` in the same directory as this notebook before running it.

In [ ]:
# Install dependencies if your workbench image does not already include them.
# Restart the kernel after this cell only if your environment requires it.

%pip install -q pandas scikit-learn torch onnx onnxruntime

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import onnx
import onnxruntime as ort

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CSV_PATH = Path("Iris.csv")
assert CSV_PATH.exists(), f"{CSV_PATH} was not found. Put Iris.csv beside the notebook."

print("PyTorch:", torch.__version__)
print("ONNX:", onnx.__version__)
print("ONNX Runtime:", ort.__version__)

In [ ]:
# Load the classic Kaggle/UCI Iris CSV.
# Expected Kaggle columns:
# Id, SepalLengthCm, SepalWidthCm, PetalLengthCm, PetalWidthCm, Species

df = pd.read_csv(CSV_PATH)

print(df.head())
print()
print(df.columns.tolist())
print()
print(df["Species"].value_counts())

In [ ]:
FEATURE_COLUMNS = [
    "SepalLengthCm",
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm",
]

CLASS_NAMES = [
    "Iris-setosa",
    "Iris-versicolor",
    "Iris-virginica",
]

class_to_id = {name: i for i, name in enumerate(CLASS_NAMES)}
id_to_class = {i: name for name, i in class_to_id.items()}

X = df[FEATURE_COLUMNS].astype("float32").to_numpy()
y = df["Species"].map(class_to_id).astype("int64").to_numpy()

if np.isnan(y).any():
    unknown = sorted(set(df.loc[pd.isna(df["Species"].map(class_to_id)), "Species"]))
    raise ValueError(f"Unexpected Species values: {unknown}")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

# Calculate normalization statistics using training data only.
train_mean = X_train.mean(axis=0).astype("float32")
train_std = X_train.std(axis=0).astype("float32")

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Mean:", train_mean)
print("Std :", train_std)

In [ ]:
class IrisMLP(nn.Module):
    def __init__(self, hidden_dim, mean, std):
        super().__init__()

        # Buffers are embedded into the exported ONNX graph.
        # Clients can therefore send the original/raw Iris measurements.
        self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32))
        self.register_buffer("std", torch.tensor(std, dtype=torch.float32))

        self.fc1 = nn.Linear(4, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 3)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = (x - self.mean) / self.std
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return self.softmax(x)

In [ ]:
def train_model(hidden_dim, seed=42, epochs=600, lr=0.02):
    # Make each version reproducible.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = IrisMLP(hidden_dim, train_mean, train_std)

    x_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.long)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()

        # CrossEntropyLoss conventionally expects logits.
        # fc2 logits are used for training; softmax is reserved for exported inference.
        x_norm = (x_tensor - model.mean) / model.std
        logits = model.fc2(model.relu(model.fc1(x_norm)))

        loss = loss_fn(logits, y_tensor)
        loss.backward()
        optimizer.step()

    model.eval()
    return model


def predict_probabilities(model, X_values):
    with torch.no_grad():
        probs = model(torch.tensor(X_values, dtype=torch.float32))
    return probs.cpu().numpy()


def evaluate_model(model, label):
    probs = predict_probabilities(model, X_test)
    pred = probs.argmax(axis=1)

    print(f"{label} accuracy: {accuracy_score(y_test, pred):.4f}")
    print(
        classification_report(
            y_test,
            pred,
            target_names=CLASS_NAMES,
            digits=4,
        )
    )
    return probs, pred

In [ ]:
# Train two genuine model versions with the same inference contract.

model_v1 = train_model(hidden_dim=8, seed=42)
model_v2 = train_model(hidden_dim=12, seed=42)

probs_v1, pred_v1 = evaluate_model(model_v1, "v1 (hidden_dim=8)")
probs_v2, pred_v2 = evaluate_model(model_v2, "v2 (hidden_dim=12)")

In [ ]:
# Compare a few predictions.
# The predicted class may often be identical while probabilities differ.

comparison = pd.DataFrame({
    "actual": [id_to_class[int(i)] for i in y_test],
    "v1_prediction": [id_to_class[int(i)] for i in pred_v1],
    "v1_confidence": probs_v1.max(axis=1),
    "v2_prediction": [id_to_class[int(i)] for i in pred_v2],
    "v2_confidence": probs_v2.max(axis=1),
})

comparison.head(15)

In [ ]:
def export_onnx(model, output_path, model_version):
    output_path = Path(output_path)

    dummy_input = torch.tensor(
        [[5.1, 3.5, 1.4, 0.2]],
        dtype=torch.float32,
    )

    torch.onnx.export(
        model,
        dummy_input,
        str(output_path),
        input_names=["input"],
        output_names=["probabilities"],
        dynamic_axes={
            "input": {0: "batch"},
            "probabilities": {0: "batch"},
        },
        opset_version=17,
        do_constant_folding=True,
    )

    # Add useful ONNX metadata without changing inference behavior.
    onnx_model = onnx.load(str(output_path))
    onnx_model.model_version = model_version

    metadata = {
        "model_name": "iris-classifier",
        "model_version": str(model_version),
        "dataset": "Kaggle uciml/iris Iris.csv",
        "features": ",".join(FEATURE_COLUMNS),
        "classes": ",".join(CLASS_NAMES),
        "framework": "PyTorch",
    }

    for key, value in metadata.items():
        entry = onnx_model.metadata_props.add()
        entry.key = key
        entry.value = value

    onnx.checker.check_model(onnx_model)
    onnx.save(onnx_model, str(output_path))

    print(f"Wrote {output_path}")


export_onnx(model_v1, "iris_v1.onnx", model_version=1)
export_onnx(model_v2, "iris_v2.onnx", model_version=2)

In [ ]:
# Inspect the ONNX operators.
# Important: you should NOT see ai.onnx.ml.TreeEnsembleClassifier.

for filename in ["iris_v1.onnx", "iris_v2.onnx"]:
    model = onnx.load(filename)

    ops = sorted({
        (node.domain or "ai.onnx", node.op_type)
        for node in model.graph.node
    })

    print(f"\n{filename}")
    print("producer:", model.producer_name)
    print("model_version:", model.model_version)
    print("operators:")
    for domain, op in ops:
        print(f"  {domain}: {op}")

    unsupported_tree_op = any(
        node.domain == "ai.onnx.ml" and node.op_type == "TreeEnsembleClassifier"
        for node in model.graph.node
    )

    print("contains TreeEnsembleClassifier:", unsupported_tree_op)

In [ ]:
# Validate both exported files with ONNX Runtime.
# This also confirms the exported model accepts raw 4-feature Iris rows.

sample = np.array(
    [
        [5.1, 3.5, 1.4, 0.2],  # typical setosa
        [6.0, 2.9, 4.5, 1.5],  # typical versicolor
        [6.7, 3.1, 5.6, 2.4],  # typical virginica
    ],
    dtype=np.float32,
)

for filename in ["iris_v1.onnx", "iris_v2.onnx"]:
    session = ort.InferenceSession(
        filename,
        providers=["CPUExecutionProvider"],
    )

    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    result = session.run(
        [output_name],
        {input_name: sample},
    )[0]

    print(f"\n{filename}")
    print("input :", input_name, session.get_inputs()[0].shape)
    print("output:", output_name, session.get_outputs()[0].shape)

    for row, probs in zip(sample, result):
        pred_id = int(np.argmax(probs))
        print(
            row.tolist(),
            "->",
            id_to_class[pred_id],
            np.round(probs, 4).tolist(),
        )

## Prepare files for an OpenVINO Model Server ModelCar

OpenVINO Model Server expects a numeric model-version directory. For the RHOAI
Model Registry demonstration, it is simplest to build one OCI artifact per registered
model version.

Each image can therefore contain:

```text
/models/
└── 1/
    └── model.onnx
```

The `1` here is the **OpenVINO serving version inside that image**. Your RHOAI Model
Registry can independently identify the OCI artifacts as model versions `1.0` and `2.0`.

In [ ]:
from pathlib import Path
import shutil

for directory in ["modelcar-v1", "modelcar-v2"]:
    path = Path(directory)
    if path.exists():
        shutil.rmtree(path)
    (path / "models" / "1").mkdir(parents=True)

shutil.copy2("iris_v1.onnx", "modelcar-v1/models/1/model.onnx")
shutil.copy2("iris_v2.onnx", "modelcar-v2/models/1/model.onnx")

print("Created:")
print("  modelcar-v1/models/1/model.onnx")
print("  modelcar-v2/models/1/model.onnx")

In [ ]:
containerfile_v1 = '''FROM registry.access.redhat.com/ubi9/ubi-minimal

RUN mkdir -p /models/1
COPY --chown=0:0 models/1/model.onnx /models/1/model.onnx
RUN chmod -R a=rX /models

LABEL org.opencontainers.image.title="Iris classification model"
LABEL org.opencontainers.image.description="Iris PyTorch MLP v1, hidden_dim=8, exported to ONNX"
LABEL org.opencontainers.image.version="1.0"
'''

containerfile_v2 = '''FROM registry.access.redhat.com/ubi9/ubi-minimal

RUN mkdir -p /models/1
COPY --chown=0:0 models/1/model.onnx /models/1/model.onnx
RUN chmod -R a=rX /models

LABEL org.opencontainers.image.title="Iris classification model"
LABEL org.opencontainers.image.description="Iris PyTorch MLP v2, hidden_dim=12, exported to ONNX"
LABEL org.opencontainers.image.version="2.0"
'''

Path("modelcar-v1/Containerfile").write_text(containerfile_v1)
Path("modelcar-v2/Containerfile").write_text(containerfile_v2)

print(Path("modelcar-v1/Containerfile").read_text())
print(Path("modelcar-v2/Containerfile").read_text())

## Example Podman build/push commands

Substitute your own registry/repository as appropriate.

```bash
podman build -t quay.io/ajblum/iris:1.0 modelcar-v1
podman push quay.io/ajblum/iris:1.0

podman build -t quay.io/ajblum/iris:2.0 modelcar-v2
podman push quay.io/ajblum/iris:2.0
```

The corresponding OCI model URIs are:

```text
oci://quay.io/ajblum/iris:1.0
oci://quay.io/ajblum/iris:2.0
```

In RHOAI, deploy these with:

- **Model type:** Predictive model
- **Model format:** ONNX
- **Serving runtime:** OpenVINO Model Server

Both versions accept a float32 tensor shaped `[batch, 4]` in this feature order:

```text
SepalLengthCm, SepalWidthCm, PetalLengthCm, PetalWidthCm
```

and return `[batch, 3]` probabilities ordered as:

```text
Iris-setosa, Iris-versicolor, Iris-virginica
```